# 01. KIND 상장폐지 목록과 우리 명단 대조 (A14)

**이 노트북이 하는 일:** 한국거래소 KIND의 상장폐지 기록(2019~2024년, 338건)을 우리 기업 명단(4,009곳)과 **이름으로 대조**해서, "우리 명단이 폐지된 기업을 얼마나 담고 있는가"를 셉니다. 이 결과가 A9(표본에서 폐지 기업을 어떻게 다룰지)의 근거이고, A14(명단 완전성)의 답입니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/raw/kind/KIND_상장폐지현황_2019-01_2024-12.tsv`(KIND 원자료 338행), `outputs/01_파일럿/기업개황_상장사전체.csv`, `상장사_명단_전체.csv` |
| 출력 | 화면 요약. `SAVE_FILES=True`일 때만 `outputs/01_파일럿/KIND_상장폐지목록_2019_2024_명단대조.csv` 저장 |
| 외부 접속 | **없음.** 이미 저장된 원자료만 읽음 |
| 실행 시간 | 약 3초 |
| 먼저 알아야 할 것 | `src/01_파일럿/01_명단_기업개황.ipynb` (명단이 어떻게 만들어졌는지) |

## 이 노트북의 지도
1. 원자료를 어떻게 얻었나 (재현 방법)
2. 원자료 읽기와 검증
3. 폐지 사유를 종류별로 나누기
4. 우리 명단과 이름으로 대조
5. 결과 읽기
6. 못 찾은 기업과 부분 일치 후보
7. 제조업 기업의 이탈 시기
8. 명단 자체의 단서: 법인구분 × 최종변경일자
9. 저장과 한계

> **이 대조는 "이름이 같다"는 것만 봅니다.** 종목코드가 KIND 목록에 없어서 회사명으로 맞췄습니다. 동명이인 오일치와 이름 변경 누락이 모두 가능합니다.

In [1]:
import re
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parents[1]                       # Corp 폴더 (이 노트북은 Corp/src/02_KIND_폐지대조에서 실행)
RAW_KIND = BASE / "data" / "raw" / "kind" / "KIND_상장폐지현황_2019-01_2024-12.tsv"
OUT = BASE / "outputs" / "01_파일럿"
SAVE_FILES = False        # True로 바꾸면 대조 결과를 csv로 저장함(기존 파일이 있으면 덮어씀)

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
print("Corp 폴더:", BASE.name, "| 원자료 존재:", RAW_KIND.exists())

Corp 폴더: Corp | 원자료 존재: True


---
# 1. 원자료를 어떻게 얻었나 (재현 방법)

2026-11-04에 사용자 허락을 받고 **KIND 웹페이지를 화면에서 읽어서** 얻었습니다. 파일을 내려받지 않았습니다.

1. `https://kind.krx.co.kr/investwarn/delcompany.do?method=searchDelCompanyMain` (KIND → 상장폐지현황)를 엽니다.
2. 시장구분 "전체", 기간 **2019-01-01 ~ 2024-12-31**로 놓고 **검색**을 누릅니다. 결과는 **338건, 23쪽**(15건씩)입니다.
3. 브라우저 개발자 도구 콘솔에서 아래 코드를 실행해 23쪽을 차례로 넘기며 표를 모읍니다(페이지의 `fnPageGo` 함수를 이용).
4. 모은 338행을 탭으로 구분한 텍스트로 저장한 것이 `data/raw/kind/KIND_상장폐지현황_2019-01_2024-12.tsv`입니다. 원래 표의 칸은 `번호, 회사명, 폐지일자, 폐지사유, 비고`이고, 회사명 칸의 아이콘에서 시장구분(유가증권/코스닥/코넥스)을 읽었습니다.

```javascript
(async () => {
  const sleep = ms => new Promise(r => setTimeout(r, ms));
  const grab = () => [...document.querySelectorAll('table.list tr')].slice(1).map(r => {
    const c = r.cells; if (c.length < 5) return null;
    const imgs = [...c[1].querySelectorAll('img')].map(i => i.alt);   // 시장구분 아이콘
    const a = c[1].querySelector('a');
    return {no: c[0].innerText.trim(), name: (a ? a.innerText : c[1].innerText).trim(),
            market: imgs[0] || '', date: c[2].innerText.trim(),
            reason: c[3].innerText.trim().replace(/\s+/g,' '), note: c[4].innerText.trim()};
  }).filter(Boolean);
  window.__all = [];
  for (let p = 1; p <= 23; p++) {            // 23쪽
    if (p > 1) { fnPageGo(String(p)); await sleep(1800); }
    window.__all.push(...grab());
  }
  return window.__all.length;                // 338이 나와야 함
})()
```

- **KIND의 이용 조건은 확인하지 않았습니다.** 논문에 쓰기 전에 출처 표기와 이용 조건을 확인해야 합니다.
- 이 단계는 KIND 페이지 구조가 바뀌면 동작하지 않을 수 있고, 다시 하려면 외부 사이트 접속이므로 허락이 필요합니다. 아래부터는 **저장된 원자료만** 씁니다.

---
# 2. 원자료 읽기와 검증

In [2]:
k = pd.read_csv(RAW_KIND, sep="\t", dtype=str, keep_default_na=False)
k["번호"] = k["번호"].astype(int)

print("읽은 행:", len(k), "(기대값 338)")
print("번호가 1~338로 빠짐없이 이어지는가:", sorted(k["번호"]) == list(range(1, 339)))
print("폐지일자 범위:", k["폐지일자"].min(), "~", k["폐지일자"].max())
print()
print("시장구분:", k["시장구분"].value_counts().to_dict())
k.head(5)

읽은 행: 338 (기대값 338)
번호가 1~338로 빠짐없이 이어지는가: True
폐지일자 범위: 2019-01-02 ~ 2024-12-26

시장구분: {'코스닥': 196, '코넥스': 102, '유가증권': 40}


,번호,회사명,시장구분,폐지일자,폐지사유,비고
0,338,하나금융21호스팩,코스닥,2024-12-26,피흡수합병(스팩소멸합병),
1,337,에스엘에너지,코스닥,2024-12-23,기업의 계속성 및 경영의 투명성 등을 종합적으로 고려하여 상장폐지기준에 해당한다고 결정,
2,336,에스엔유,코스닥,2024-12-20,피흡수합병,
3,335,듀켐바이오,코넥스,2024-12-20,코스닥시장 이전상장,듀켐바이오
4,334,하나금융22호스팩,코스닥,2024-12-19,피흡수합병(스팩소멸합병),


### 해석
- 338행이 빠짐없이 읽혔고 번호가 1~338로 연속입니다. 번호가 연속이라는 것은 쪽을 넘기며 모으다가 **빠뜨린 쪽이 없다**는 확인이지, KIND 자료 자체가 완전하다는 확인은 아닙니다.
- 시장구분은 **KIND가 폐지 당시의 시장**을 표시한 것입니다. 우리 명단의 `corp_cls`(현재 법인구분)와는 시점이 다릅니다.
- `비고` 칸은 KIND가 적은 **변경 후 상호**입니다(예: 코넥스에서 코스닥으로 옮기며 이름이 바뀐 경우). 이름이 바뀌면 DART 명단의 이름과 달라서 대조에 쓰입니다.

---
# 3. 폐지 사유를 종류별로 나누기

폐지 사유는 문장으로 적혀 있어서 그대로는 셀 수 없습니다. 문장에 들어 있는 말로 **규칙을 만들어 분류**합니다. 이 분류는 **제가 만든 것이며 거래소의 공식 분류가 아닙니다.**

중요한 구분은 두 가지입니다.
- **시장 이전**: 코넥스→코스닥, 코스닥→유가증권처럼 다른 시장에 다시 상장한 것. **실제 퇴출이 아닙니다.** 이름만 목록에서 사라집니다.
- **스팩**: 합병 목적의 페이퍼컴퍼니. 제조업 분석과 무관합니다.

In [3]:
def classify(r):
    n, s = r["회사명"], r["폐지사유"]
    if s.startswith("신한제9호기업인수목적"):
        return "합병(피흡수)"
    if "이전상장" in s or re.search(r"(유가증권|코스닥)시장 상장$", s):
        return "시장 이전(다른 시장 재상장)"
    if "스팩" in n or "SPAC" in n or "예비심사" in s or "스팩소멸합병" in s:
        return "스팩(합병·미제출 등)"
    if "피흡수합병" in s:
        return "합병(피흡수)"
    if "완전자회사" in s or "주식교환" in s:
        return "완전자회사화·주식교환"
    if "감사의견" in s or "감사범위" in s:
        return "감사의견(거절·부적정·한정)"
    if "기업의 계속성" in s:
        return "실질심사(계속성·투명성)"
    if "해산" in s or "파산" in s or "존속기간" in s:
        return "해산·파산·존속만료"
    if "상장폐지 신청" in s or "신청에 의한" in s:
        return "자진 상장폐지 신청"
    if "지정자문인" in s or "분산요건" in s:
        return "코넥스 요건 미충족"
    if "어음" in s:
        return "부도·거래정지"
    if "미제출" in s:
        return "공시 미제출"
    return "기타"

k["분류"] = k.apply(classify, axis=1)
k["분류"].value_counts().to_frame("건수")

,건수
분류,
스팩(합병·미제출 등),79
감사의견(거절·부적정·한정),58
시장 이전(다른 시장 재상장),51
합병(피흡수),37
실질심사(계속성·투명성),27
완전자회사화·주식교환,27
해산·파산·존속만료,18
코넥스 요건 미충족,17
자진 상장폐지 신청,12


### 해석
- **순서가 중요한 규칙**입니다. 예를 들어 "피흡수합병(스팩소멸합병)"은 스팩 이름이면 "스팩"으로, 일반 회사면 "합병(피흡수)"으로 갑니다. 맨 위의 `신한제9호기업인수목적…` 한 줄은 일반 회사(다원넥스뷰)가 스팩에 흡수된 특수한 경우라 따로 처리했습니다.
- 가장 많은 것은 **스팩**과 **감사의견 문제**입니다. 시장 이전은 51건으로 실제 퇴출이 아니므로 이후 계산에서 뺍니다.
- 분류에서 **"기타" 2건**은 사업보고서 미제출 사유가 길게 적힌 건입니다. 판정을 어렵게 만드는 것은 사유 문장이 여러 가지 이유를 한꺼번에 적은 경우입니다.

---
# 4. 우리 명단과 이름으로 대조

대조 규칙:
1. 이름에서 `(주)`, `주식회사`, `㈜`와 **공백을 모두 지우고 대문자로** 맞춥니다.
2. KIND의 `회사명`이 우리 명단의 `corp_name`과 같으면 일치.
3. 같지 않으면 KIND의 `비고`(변경 후 상호)로 한 번 더 찾습니다.
4. 우리 명단에서 같은 이름이 **둘 이상**이면 "동명 N곳"으로 표시합니다(어느 쪽인지 모르므로 오일치 가능).

In [4]:
c = pd.read_csv(OUT / "기업개황_상장사전체.csv", dtype=str, keep_default_na=False)
l = pd.read_csv(OUT / "상장사_명단_전체.csv", dtype=str, keep_default_na=False)
c = c.merge(l[["corp_code", "modify_date"]], on="corp_code", how="left")

def norm(x):
    x = re.sub(r"\(주\)|주식회사|㈜", "", x)
    return re.sub(r"\s+", "", x).upper()

c["key"] = c["corp_name"].map(norm)
by = {key: g for key, g in c.groupby("key")}
print("우리 명단 4,009곳 중 이름이 겹치는 이름 수:", int((c["key"].value_counts() > 1).sum()))

res = []
for _, r in k.iterrows():
    key1 = norm(r["회사명"]); key2 = norm(r["비고"]) if r["비고"] else ""
    hit, how = None, ""
    if key1 in by:
        hit, how = by[key1], "회사명"
    elif key2 and key2 in by:
        hit, how = by[key2], "변경상호(비고)"
    if hit is None:
        res.append(("N", "", "", "", "", "", ""))
    else:
        h = hit.iloc[0]
        res.append(("Y" if len(hit) == 1 else f"Y(동명 {len(hit)}곳)", how, h["corp_code"], h["corp_cls"],
                    h["induty_code"], h["modify_date"], h["stock_code"]))
k[["DART명단_일치", "일치방식", "corp_code", "현재_법인구분", "업종코드", "명단_최종변경일자", "종목코드"]] = pd.DataFrame(res, index=k.index)
k["제조업(앞2자리10~34)"] = k["업종코드"].map(lambda v: "" if not v else ("Y" if "10" <= v[:2] <= "34" else "N"))
k["일치여부"] = k["DART명단_일치"].str[0]
print("명단에서 이름이 일치한 건:", int((k["일치여부"] == "Y").sum()), "/", len(k))
print("일치 방식:", k.loc[k["일치여부"] == "Y", "일치방식"].value_counts().to_dict())
print("동명 중복으로 표시된 일치:", int(k["DART명단_일치"].str.contains("동명").sum()))

우리 명단 4,009곳 중 이름이 겹치는 이름 수: 30
명단에서 이름이 일치한 건: 204 / 338
일치 방식: {'회사명': 194, '변경상호(비고)': 10}
동명 중복으로 표시된 일치: 12


---
# 5. 결과 읽기

In [5]:
print("[분류별 건수 × 우리 명단 일치]")
display(pd.crosstab(k["분류"], k["일치여부"], margins=True))

real = k[~k["분류"].isin(["시장 이전(다른 시장 재상장)", "스팩(합병·미제출 등)"])]
m = real[real["일치여부"] == "Y"]
print(f"\n실제 상장 종료(시장 이전·스팩 제외): {len(real)}건 중 명단 일치 {len(m)}건 ({len(m)/len(real):.0%})")
print("일치한 건의 현재 법인구분:", m["현재_법인구분"].value_counts().to_dict())

[분류별 건수 × 우리 명단 일치]


일치여부,N,Y,All
분류,,,
감사의견(거절·부적정·한정),11,47,58
공시 미제출,1,5,6
기타,2,0,2
부도·거래정지,1,3,4
스팩(합병·미제출 등),79,0,79
시장 이전(다른 시장 재상장),1,50,51
실질심사(계속성·투명성),9,18,27
완전자회사화·주식교환,10,17,27
자진 상장폐지 신청,7,5,12



실제 상장 종료(시장 이전·스팩 제외): 208건 중 명단 일치 154건 (74%)
일치한 건의 현재 법인구분: {'E': 150, 'K': 4}


### 해석
- **시장 이전 51건 중 50건이 일치**하고, 일치한 곳은 현재도 유가증권(Y)·코스닥(K)으로 남아 있습니다. 이들은 실제로 퇴출된 것이 아니라서 **분석 대상에서 빠지면 안 되는 기업**입니다. KIND의 "폐지 목록"만 보고 이들을 이탈로 세면 틀립니다.
- **스팩 79건은 하나도 일치하지 않습니다.** 우리 명단에서 스팩의 이름이 "○○호기업인수목적" 형태라 KIND의 "○○호스팩"과 달라서 대조가 되지 않은 것으로 **보입니다.** 이 이유를 확인하지는 않았습니다. 제조업도 아니라서 A9와 무관합니다.
- **실제 상장 종료 208건 중 154건(74%)이 명단에서 확인됩니다.** 그리고 일치한 154건 중 **150건의 현재 법인구분이 E**입니다. 즉 **폐지 기업의 대부분이 E로 분류되어 명단에 남아 있습니다.** 다만 OpenDART 가이드에서 `corp_cls` E는 "기타"라는 뜻일 뿐 상장폐지를 뜻하는 코드가 아니므로, **E인 기업이 모두 폐지 기업이라는 뜻은 아닙니다**(방향이 거꾸로입니다: 폐지 기업 → 대부분 E).
- 나머지 4건(K)은 폐지 기업과 **이름이 같은 다른 회사**일 가능성이 있습니다. 이런 오일치는 이 방법으로 피할 수 없습니다.

In [6]:
k["폐지연도"] = k["폐지일자"].str[:4]
r = real.assign(폐지연도=real["폐지일자"].str[:4])
t = pd.crosstab(r["폐지연도"], r["일치여부"])
t["일치율"] = (t["Y"] / (t["Y"] + t["N"])).round(2)
t

일치여부,N,Y,일치율
폐지연도,,,
2019,5,20,0.80
2020,14,26,0.65
2021,9,29,0.76
2022,7,31,0.82
2023,7,21,0.75
2024,12,27,0.69


### 해석
연도별 일치율은 65~82% 사이에서 오르내리고 **최근 폐지일수록 낮아지는 뚜렷한 추세가 없습니다.** 최근에 폐지된 기업이 명단에서 더 빠진다는 증거는 아닙니다. 각 연도의 건수가 40건 안팎이라 한두 건에 비율이 크게 흔들립니다.

---
# 6. 못 찾은 기업과 부분 일치 후보

스팩·시장 이전을 뺀 실제 상장 종료 중 **이름으로 못 찾은 건**입니다. 이 중에는 DART 정식 이름이 더 길어서 못 찾은 경우가 섞여 있을 수 있어서, **한쪽 이름이 다른 쪽에 들어 있는지**(부분 일치)도 봅니다.

In [7]:
un = real[real["일치여부"] == "N"].copy()
print("이름으로 못 찾은 건:", len(un))

rows = []
for _, r in un.iterrows():
    a = norm(r["회사명"])
    cand = c[c["key"].map(lambda s: len(a) >= 3 and (a in s or (len(s) >= 3 and s in a)))]
    rows.append({"번호": r["번호"], "회사명": r["회사명"], "폐지일자": r["폐지일자"], "분류": r["분류"],
                 "부분 일치 후보": "; ".join(f"{x.corp_name}({x.corp_cls},{x.induty_code})" for x in cand.itertuples())[:90]})
cand_df = pd.DataFrame(rows)
print("부분 일치 후보가 있는 건:", int((cand_df["부분 일치 후보"] != "").sum()), "/ 없는 건:", int((cand_df["부분 일치 후보"] == "").sum()))
cand_df[cand_df["부분 일치 후보"] != ""]

이름으로 못 찾은 건: 54


부분 일치 후보가 있는 건: 21 / 없는 건: 33


,번호,회사명,폐지일자,분류,부분 일치 후보
0,337,에스엘에너지,2024-12-23,실질심사(계속성·투명성),"에스엘(Y,303)"
1,336,에스엔유,2024-12-20,합병(피흡수),"에스엔유프리시젼(E,292)"
2,329,비즈니스온,2024-11-22,완전자회사화·주식교환,"비즈니스온커뮤니케이션(E,58222)"
3,321,동아타이어,2024-10-08,해산·파산·존속만료,"동아타이어공업(E,221)"
6,303,대양제지,2024-06-21,자진 상장폐지 신청,"신대양제지(Y,172); 대양제지공업(E,1710)"
8,294,아리온,2024-05-03,감사의견(거절·부적정·한정),"아리온테크놀로지(E,264)"
10,283,골프존데카,2024-02-22,완전자회사화·주식교환,"골프존(K,58219)"
12,267,SBS콘텐츠허브,2023-12-18,완전자회사화·주식교환,"SBS(Y,60210)"
13,263,바다로19호,2023-11-24,자진 상장폐지 신청,"바다로19호선박투자회사(E,76190)"
16,245,데이드림엔터,2023-08-10,자진 상장폐지 신청,"데이드림엔터테인먼트(E,591)"


### 해석
- **후보가 있는 건**은 **같은 회사로 보이는 것**과 **다른 회사일 가능성이 큰 것**이 섞여 있습니다.
  - 같은 회사로 보임(DART 정식명이 더 긺): 맘스터치 → 맘스터치앤컴퍼니, 동아타이어 → 동아타이어공업, 아리온 → 아리온테크놀로지, 데이드림엔터 → 데이드림엔터테인먼트, 바다로19호 → …선박투자회사
  - 다른 회사일 가능성이 큼: 에스엘에너지 → 에스엘, SKC 솔믹스 → SKC, 골프존데카 → 골프존, SBS콘텐츠허브 → SBS, 에이치디 → 에이치디현대인프라코어
- 이것은 **이름만 본 판단**입니다. 어느 쪽인지는 **DART 공시 검색에서 확인**해야 하고, 확인하기 전에는 일치로 세지 않았습니다.
- **후보도 없는 건**(약 33건)은 명단에 없을 가능성이 있습니다. 명단에 없으면 업종도 알 수 없어서, 그 기업이 제조업인지 모릅니다.

In [8]:
cand_df[cand_df["부분 일치 후보"] == ""][["번호", "회사명", "폐지일자", "분류"]]

,번호,회사명,폐지일자,분류
4,312,녹원씨엔아이,2024-07-29,실질심사(계속성·투명성)
5,306,파나케이아,2024-06-25,실질심사(계속성·투명성)
7,301,THE MIDONG,2024-06-17,실질심사(계속성·투명성)
9,284,DL건설,2024-03-04,완전자회사화·주식교환
11,278,SK렌터카,2024-01-31,완전자회사화·주식교환
14,257,코스온,2023-10-20,감사의견(거절·부적정·한정)
15,249,우리종금,2023-08-28,완전자회사화·주식교환
18,211,베트남개발1,2023-02-02,해산·파산·존속만료
19,196,뉴로스,2022-10-26,감사의견(거절·부적정·한정)
20,193,연이비앤티,2022-10-17,실질심사(계속성·투명성)


---
# 7. 제조업 기업의 이탈 시기

명단에서 이름이 일치했고 업종 앞 2자리가 10~34인 **실제 상장 종료**를, 폐지일로 시기를 나눠 셉니다. A9에서 사전기간은 2019Q1~2021Q2입니다.

In [9]:
mm = m[m["제조업(앞2자리10~34)"] == "Y"].copy()

def period(d):
    if d <= "2021-06-30":
        return "사전기간 중(2019-01~2021-06)"
    if d <= "2022-12-31":
        return "인상기 초(2021-07~2022-12)"
    return "2023~2024"

mm["기간"] = mm["폐지일자"].map(period)
print("명단에서 확인된 제조업 실제 상장 종료:", len(mm), "건")
print(mm["기간"].value_counts().to_dict())
print("현재 법인구분:", mm["현재_법인구분"].value_counts().to_dict(), "| 동명 중복 표시:", int(mm["DART명단_일치"].str.contains("동명").sum()))
pd.crosstab(mm["기간"], mm["분류"])

명단에서 확인된 제조업 실제 상장 종료: 89 건
{'사전기간 중(2019-01~2021-06)': 39, '2023~2024': 28, '인상기 초(2021-07~2022-12)': 22}
현재 법인구분: {'E': 85, 'K': 4} | 동명 중복 표시: 9


분류,감사의견(거절·부적정·한정),공시 미제출,부도·거래정지,실질심사(계속성·투명성),완전자회사화·주식교환,자진 상장폐지 신청,코넥스 요건 미충족,합병(피흡수),해산·파산·존속만료
기간,,,,,,,,,
2023~2024,9,0,1,6,5,1,2,3,1
사전기간 중(2019-01~2021-06),9,2,0,4,2,0,6,11,5
인상기 초(2021-07~2022-12),9,1,2,2,1,1,0,5,1


In [10]:
# 금리 인상 시작(2021-08-26)부터 분석 종료(2024-09-30)까지로 다시 센다
w = mm[(mm["폐지일자"] >= "2021-08-26") & (mm["폐지일자"] <= "2024-09-30")]
w2 = w[w["시장구분"] != "코넥스"]
after = mm[mm["폐지일자"] > "2024-09-30"]
dup = lambda d: int(d["DART명단_일치"].str.contains("동명").sum())

print("제조업 실제 종료 89건의 폐지 당시 시장:", mm["시장구분"].value_counts().to_dict())
print(f"2021-08-26 ~ 2024-09-30 폐지: {len(w)}건 (동명 표시 {dup(w)}건)")
print(f"  그중 코넥스 제외: {len(w2)}건 (동명 표시 {dup(w2)}건)")
print(f"분석 종료(2024-09-30) 뒤에 폐지된 건: {len(after)}건")
print("  ", after[["회사명", "폐지일자", "분류"]].values.tolist())
print("2021-08-26~2024-09-30 39건의 분류:", w["분류"].value_counts().to_dict())

제조업 실제 종료 89건의 폐지 당시 시장: {'코스닥': 45, '코넥스': 34, '유가증권': 10}
2021-08-26 ~ 2024-09-30 폐지: 39건 (동명 표시 2건)
  그중 코넥스 제외: 25건 (동명 표시 1건)
분석 종료(2024-09-30) 뒤에 폐지된 건: 7건
   [['락앤락', '2024-12-09', '완전자회사화·주식교환'], ['티엘아이', '2024-11-14', '실질심사(계속성·투명성)'], ['제이시스메디칼', '2024-11-07', '완전자회사화·주식교환'], ['어스앤에어로스페이스', '2024-11-06', '실질심사(계속성·투명성)'], ['이루다', '2024-10-22', '합병(피흡수)'], ['한국테크놀로지', '2024-10-07', '해산·파산·존속만료'], ['씨엔티드림', '2024-10-07', '감사의견(거절·부적정·한정)']]
2021-08-26~2024-09-30 39건의 분류: {'감사의견(거절·부적정·한정)': 15, '실질심사(계속성·투명성)': 6, '합병(피흡수)': 5, '완전자회사화·주식교환': 4, '부도·거래정지': 3, '코넥스 요건 미충족': 2, '자진 상장폐지 신청': 2, '해산·파산·존속만료': 1, '공시 미제출': 1}


### 해석
- 제조업 실제 종료가 **89건**이고 **사전기간 중 39건 / 인상기 초 22건 / 2023~2024년 28건**입니다.
- A9 규칙으로 보면 **사전기간 중에 폐지된 39건은 사전기간 전체를 채우지 못해** 표본에 들어오기 어렵습니다. 사전기간이 끝난 뒤(2021-07 이후) 사라진 **50건(22+28)** 이 "금리 인상기 이후 이탈"의 후보입니다. 
- 이탈 사유는 **감사의견 문제 27건(거절·부적정 등)** 이 가장 많고, 합병(피흡수) 19건, 실질심사 12건 순입니다. **합병으로 사라진 기업이 "부실 기업"이라는 보장은 없습니다.** 그래서 A9에서 이탈의 **방향을 단정하지 않기로** 했습니다.
- 이 89건은 **명단에서 이름이 확인된 기업뿐**입니다. 명단에 없는 기업은 업종을 몰라 센 적이 없습니다.
- **89건 중 코넥스가 34건**(코스닥 45, 유가증권 10)입니다. A2에서 코넥스를 제외하기로 하면 수가 달라집니다. 인상 시작~분석 종료 폐지 39건은 코넥스를 빼면 25건입니다(위 출력).
- **동명 회사가 둘 이상이라 같은 법인인지 모호한 일치가 89건 중 9건**입니다. 89건 모두 동일 법인으로 확정된 것은 아닙니다.
- 폐지일이 분석 종료(2024-09-30) **뒤**인 건(7건)은 이탈로 보면 안 됩니다. 폐지 직전 분기까지 자료가 있으면 2024Q3까지 관측이 유지될 수 있어서, 이탈 여부는 **마지막 관측 분기**로 판단해야 합니다(A9 규칙 ④).
- 2021-07~2021-08-25에 폐지된 4건은 금리 인상 시작 이전이라 "인상 이후 이탈"에 넣으면 안 됩니다.

---
# 8. 명단 자체의 단서: 법인구분 × 최종변경일자

앞서 "E 1,248곳 중 576곳은 최종변경일자가 2017-06-30"이라는 점을 폐지 기업이 남아 있다는 **정황**으로 썼습니다. 이를 KIND 대조 결과와 함께 다시 봅니다.

In [11]:
c["연도"] = c["modify_date"].str[:4]
print("[우리 명단 4,009곳] 법인구분 × 최종변경일자 연도")
display(pd.crosstab(c["연도"], c["corp_cls"]))

hit = k[(k["일치여부"] == "Y") & (~k["분류"].isin(["시장 이전(다른 시장 재상장)", "스팩(합병·미제출 등)"]))]
print("KIND와 일치한 실제 상장 종료 기업의 최종변경일자 연도:")
print(hit["명단_최종변경일자"].str[:4].value_counts().sort_index().to_dict())

[우리 명단 4,009곳] 법인구분 × 최종변경일자 연도


corp_cls,E,K,N,Y
연도,,,,
2017,576,0,0,0
2018,35,1,0,0
2019,32,1,0,0
2020,39,1,0,1
2021,28,8,0,5
2022,69,161,17,73
2023,73,204,14,103
2024,122,278,12,127
2025,103,352,10,163


KIND와 일치한 실제 상장 종료 기업의 최종변경일자 연도:
{'2017': 1, '2019': 11, '2020': 15, '2021': 8, '2022': 32, '2023': 25, '2024': 36, '2025': 13, '2026': 13}


### 해석
- 법인구분 **E에서 2017년으로 갱신이 멈춘 576곳**은 KIND 폐지 목록(2019~2024)과 거의 겹치지 않습니다(위 마지막 줄에서 2017년은 1건). 즉 **2019년 이후 폐지된 기업이 아니라 그 전에 폐지·정리된 기업으로 보입니다.** 앞서 이 576곳을 "폐지 기업의 정황"으로 해석한 것은 맞는 방향이지만, **우리 분석 기간(2019~2024)의 폐지 기업과는 대부분 다른 집합**입니다.
- 2019~2024년에 폐지된 기업은 최종변경일자가 **폐지 이후 연도(2019~2026)** 로 분산되어 있습니다. 폐지될 때 DART 기업 정보가 갱신된 것으로 **추정**됩니다.
- 이 해석도 "정황"입니다. DART가 최종변경일자를 어떻게 갱신하는지는 가이드에 설명이 없습니다.

---
# 9. 저장과 한계

In [12]:
out_cols = ["번호", "회사명", "시장구분", "폐지일자", "폐지사유", "비고", "분류", "DART명단_일치", "일치방식", "corp_code",
            "현재_법인구분", "업종코드", "명단_최종변경일자", "종목코드", "제조업(앞2자리10~34)", "일치여부"]
result = k[out_cols]
saved_path = OUT / "KIND_상장폐지목록_2019_2024_명단대조.csv"

if saved_path.exists():
    old = pd.read_csv(saved_path, dtype=str, keep_default_na=False)
    same = old[out_cols].reset_index(drop=True).equals(result.astype(str).reset_index(drop=True))
    print("이미 저장된 대조 결과 csv와 같은가:", same)

if SAVE_FILES:
    result.to_csv(saved_path, index=False, encoding="utf-8-sig")
    print("저장:", saved_path.name)
else:
    print("SAVE_FILES=False → 저장하지 않았습니다.")

이미 저장된 대조 결과 csv와 같은가: True
SAVE_FILES=False → 저장하지 않았습니다.


### 정리

**이 노트북으로 확인된 것**
- KIND 폐지 기록 338건 중 시장 이전 51건은 실제 퇴출이 아니고, 스팩 79건은 이름 형식이 달라 대조되지 않았습니다.
- **실제 상장 종료 208건 중 154건(74%)이 우리 명단에서 이름으로 확인**되고, 그중 150건이 법인구분 E입니다. **폐지 기업이 명단에 상당수 남아 있습니다.**
- 명단에서 확인된 제조업 실제 종료 89건 중 금리 인상 시작(2021-08-26)부터 분석 종료(2024-09-30) 사이 폐지는 39건(코넥스 제외 25건)입니다. 이는 **KIND 폐지 건수이지 회귀 표본의 이탈 기업 수가 아닙니다**(사전기간 적격성·재무보고서 존재·동일 법인 여부를 아직 확인하지 않음).

**확인하지 못한 것**
| 한계 | 이유 |
|---|---|
| 이름 대조의 오일치·누락 | 종목코드가 KIND 목록에 없어 이름으로만 맞춤. 동명 12건 표시 |
| 못 찾은 54건이 이름 문제인지 진짜 누락인지 | 부분 일치 후보 21건은 DART 공시 검색으로 확인해야 함 |
| 누락된 기업의 업종 | 명단에 없으면 업종코드가 없음 |
| 명단에 있는 기업의 재무제표가 DART에 있는지 | 수집 단계에서 확인 |
| KIND 이용 조건, 폐지 사유 분류의 공식성 | 확인하지 않음. 분류는 제가 만든 규칙 |

**다음 단계:** 부분 일치 후보 21건을 DART 공시 검색으로 확인할지(A14 선택지 2), 한계로만 기록할지는 사용자 결정 대기입니다.